# `qry` — Advanced Reference

`qry` filters a DataFrame using a dict of conditions. All condition types supported:

| Syntax | Meaning |
|---|---|
| `{'col': value}` | equality |
| `{'col': [v1, v2]}` | in list |
| `{'col': ('in', [...])}` | in list (explicit) |
| `{'col': ('not in', [...])}` | exclusion |
| `{'col': ('>', n)}` | comparison operator |
| `{'col': '(a,b)'}` | open interval `a < x < b` |
| `{'col': '[a,b)'}` | half-open interval `a <= x < b` |
| `{'col': '[a,b]'}` | closed interval `a <= x <= b` |

Chain multiple `.qry()` calls to apply conditions on the same column twice.

---

In [31]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']
titanic = pt.sample_data['titanic']

## Equality and multi-value filters

In [32]:
# Single equality — keep only Dream island
# can use pandas .query but qry is more flexible for majority of use cases
(penguins
 .qry({'island': 'Dream'})
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,Dream,Female,27,36.911111,17.618519,187.851852,3344.444444
1,Adelie,Dream,Male,28,40.071429,18.839286,191.928571,4045.535714
2,Chinstrap,Dream,Female,34,46.573529,17.588235,191.735294,3527.205882
3,Chinstrap,Dream,Male,34,51.094118,19.252941,199.911765,3938.970588


In [33]:
# List — keep Adelie and Chinstrap, aggregate per species + island
(penguins
 .qry({'species': ['Adelie', 'Chinstrap']})
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,Biscoe,Female,22,37.359091,17.704545,187.181818,3369.318182
1,Adelie,Biscoe,Male,22,40.590909,19.036364,190.409091,4050.000000
2,Adelie,Dream,Female,27,36.911111,17.618519,187.851852,3344.444444
3,Adelie,Dream,Male,28,40.071429,18.839286,191.928571,4045.535714
4,Adelie,Torgersen,Female,24,37.554167,17.550000,188.291667,3395.833333
5,Adelie,Torgersen,Male,23,40.586957,19.391304,194.913043,4034.782609
6,Chinstrap,Dream,Female,34,46.573529,17.588235,191.735294,3527.205882
7,Chinstrap,Dream,Male,34,51.094118,19.252941,199.911765,3938.970588


In [34]:
# 'not in' — exclude Male penguins
(penguins
 .qry({'sex': ('not in', ['Male'])})
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,Biscoe,Female,22,37.359091,17.704545,187.181818,3369.318182
1,Adelie,Dream,Female,27,36.911111,17.618519,187.851852,3344.444444
2,Adelie,Torgersen,Female,24,37.554167,17.550000,188.291667,3395.833333
3,Chinstrap,Dream,Female,34,46.573529,17.588235,191.735294,3527.205882
4,Gentoo,Biscoe,Female,58,45.563793,14.237931,212.706897,4679.741379


## Comparison operators

In [35]:
# Supported operators: >, >=, <, <=, ==, !=
# == is the default — {'col': value} and {'col': ('==', value)} are identical
# Heavy penguins (body_mass_g > 5000) across species
(penguins
 .qry({'body_mass_g': ('>', 5000)})
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Gentoo,Biscoe,Female,5,46.1600,14.440000,213.800000,5140.000000
1,Gentoo,Biscoe,Male,56,49.5875,15.757143,221.714286,5533.928571


In [36]:
# Combine multiple column conditions in one qry call
# Gentoo females with large bills — multi-condition dict
(penguins
 .qry({'species': 'Gentoo', 'sex': 'Female', 'bill_length_mm': ('>=', 45)})
 .select(['species', 'island', 'sex', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,body_mass_g
0,Gentoo,Biscoe,Female,39,46.697436,4710.25641


## Interval notation
String intervals let you filter a range on the same column without chaining two `.qry()` calls.
`(` / `)` = exclusive bound, `[` / `]` = inclusive bound.

In [37]:
# Open interval — strictly between 3500 and 4500 g
(penguins
 .qry({'body_mass_g': '(3500,4500)'})
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,Biscoe,Female,8,37.387500,18.112500,190.875000,3743.750000
1,Adelie,Biscoe,Male,19,40.168421,18.921053,189.368421,3947.368421
2,Adelie,Dream,Female,5,38.180000,18.200000,189.800000,3620.000000
3,Adelie,Dream,Male,24,40.037500,18.891667,192.958333,4046.875000
4,Adelie,Torgersen,Female,9,37.488889,17.600000,188.111111,3669.444444
5,Adelie,Torgersen,Male,17,40.411765,19.441176,194.764706,4035.294118
6,Chinstrap,Dream,Female,19,47.310526,17.747368,191.894737,3718.421053
7,Chinstrap,Dream,Male,27,50.977778,19.222222,200.185185,3950.925926
8,Gentoo,Biscoe,Female,15,44.873333,13.973333,210.200000,4301.666667


In [38]:
# Half-open interval — bill_length_mm >= 40 and < 50
(penguins
 .qry({'bill_length_mm': '[40,50)'})
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,Biscoe,Female,1,40.500000,17.900000,187.000000,3200.000000
1,Adelie,Biscoe,Male,15,41.666667,19.120000,192.400000,4153.333333
2,Adelie,Dream,Female,2,41.200000,17.800000,186.500000,3475.000000
3,Adelie,Dream,Male,15,41.333333,18.786667,194.066667,3963.333333
4,Adelie,Torgersen,Female,4,40.625000,17.350000,186.000000,3400.000000
5,Adelie,Torgersen,Male,13,42.746154,19.023077,197.230769,4096.153846
6,Chinstrap,Dream,Female,29,45.648276,17.493103,191.517241,3516.379310
7,Chinstrap,Dream,Male,8,49.225000,18.812500,199.875000,3909.375000
8,Gentoo,Biscoe,Female,57,45.477193,14.221053,212.649123,4674.122807
9,Gentoo,Biscoe,Male,36,47.872222,15.597222,219.416667,5440.972222


In [39]:
# Closed interval — body_mass_g between 3500 and 4000 inclusive
(penguins
 .qry({'body_mass_g': '[3500,4000]'})
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,Biscoe,Female,9,37.633333,18.066667,190.333333,3716.666667
1,Adelie,Biscoe,Male,11,39.536364,18.627273,186.545455,3772.727273
2,Adelie,Dream,Female,7,37.800000,18.300000,190.142857,3585.714286
3,Adelie,Dream,Male,13,39.530769,18.792308,190.692308,3842.307692
4,Adelie,Torgersen,Female,9,37.488889,17.600000,188.111111,3669.444444
5,Adelie,Torgersen,Male,12,40.016667,19.366667,194.666667,3804.166667
6,Chinstrap,Dream,Female,20,47.245000,17.660000,191.950000,3675.000000
7,Chinstrap,Dream,Male,16,51.056250,19.237500,198.625000,3792.187500
8,Gentoo,Biscoe,Female,1,42.700000,13.700000,208.000000,3950.000000


In [40]:
# Interval notation also works on STRING columns — alphabetical comparison
# 'island' values: Biscoe, Dream, Torgersen — (Biscoe, Dream] means Dream only
(penguins
 .agg_df(aggfunc=['mean', 'n'])
 .qry({'island': '(Biscoe,Dream]'})  # all islands greater than Biscoe and up to Dream
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
2,Adelie,Dream,Female,27,36.911111,17.618519,187.851852,3344.444444
3,Adelie,Dream,Male,28,40.071429,18.839286,191.928571,4045.535714
6,Chinstrap,Dream,Female,34,46.573529,17.588235,191.735294,3527.205882
7,Chinstrap,Dream,Male,34,51.094118,19.252941,199.911765,3938.970588


In [41]:
# GOTCHA: [v1, v2] (Python list) vs '[v1,v2]' (string) are completely different
# List → isin filter (matches exactly those values)
# String with brackets → closed interval (range filter)
(penguins
 .agg_df(aggfunc=['mean', 'n'])
 .qry({'body_mass_g': [89100, 120000]})  # isin — only rows where value is exactly 89100 or 120000
 # compare with .qry({'body_mass_g': '[89100,120000]'}) — range: 89100 <= x <= 120000
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g


In [42]:
# ('in', [...]) is the explicit tuple form — identical result to passing a list directly
# Useful when you want to be explicit, or when building conditions programmatically
(penguins
 .agg_df(aggfunc=['mean', 'n'])
 .qry({'body_mass_g': ('in', [89100, 120000])})  # same as .qry({'body_mass_g': [89100, 120000]})
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g


## Same-column double filter: chain two `.qry()` calls
`dict` keys are unique so you can't write `{'col': ('>',a), 'col': ('<',b)}`.
The solution is to chain two `.qry()` calls, or use interval notation (shown above).

In [43]:
# Equivalent to: bill_length_mm > 40 AND bill_length_mm < 50
# (same result as '[40,50)' interval but with chained qry)
(penguins
 .qry({'bill_length_mm': ('>', 40)})
 .qry({'bill_length_mm': ('<', 50)})
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,sex,n,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,Biscoe,Female,1,40.500000,17.900000,187.000000,3200.000000
1,Adelie,Biscoe,Male,15,41.666667,19.120000,192.400000,4153.333333
2,Adelie,Dream,Female,2,41.200000,17.800000,186.500000,3475.000000
3,Adelie,Dream,Male,15,41.333333,18.786667,194.066667,3963.333333
4,Adelie,Torgersen,Female,4,40.625000,17.350000,186.000000,3400.000000
5,Adelie,Torgersen,Male,13,42.746154,19.023077,197.230769,4096.153846
6,Chinstrap,Dream,Female,29,45.648276,17.493103,191.517241,3516.379310
7,Chinstrap,Dream,Male,8,49.225000,18.812500,199.875000,3909.375000
8,Gentoo,Biscoe,Female,57,45.477193,14.221053,212.649123,4674.122807
9,Gentoo,Biscoe,Male,36,47.872222,15.597222,219.416667,5440.972222


## Real-world pipeline: filter → select → aggregate

In [44]:
# Tips: weekend dinner table of 2+, spending > $10 — average tip by smoker status
(tips
 .qry({'day': ['Sat', 'Sun'], 'time': 'Dinner', 'size': ('>=', 2), 'total_bill': ('>', 10)})
 .select(['smoker', 'tip', 'total_bill'])
 .agg_df(aggfunc=['mean', 'n'])
)

,smoker,n,tip,total_bill
0,Yes,57,3.087719,23.232281
1,No,97,3.225464,20.705876


In [45]:
# Titanic: survivors in 1st/2nd class, age between 18 and 40
(titanic
 .qry({'survived': 1, 'pclass': [1, 2]})
 .qry({'age': '[18,40]'})
 .select(['pclass', 'sex', 'age', 'fare'])
 .agg_df(aggfunc=['mean', 'n'])
)

,sex,n,pclass,age,fare
0,female,95,1.473684,29.057895,70.939693
1,male,26,1.153846,31.269231,79.328212
